In [1]:
!pip install git+https://github.com/neurallatents/nlb_tools.git
!pip install dandi
!dandi download https://gui.dandiarchive.org/dandiset/000127
!pip install git+https://github.com/neurallatents/nlb_tools.git --no-deps
!pip install pynwb elephant pyyaml

  Cloning https://github.com/neurallatents/nlb_tools.git to /tmp/pip-req-build-_5kebx57
  Running command git clone --filter=blob:none --quiet https://github.com/neurallatents/nlb_tools.git /tmp/pip-req-build-_5kebx57
  Resolved https://github.com/neurallatents/nlb_tools.git to commit 42f8410b88e12db136910fa2f888b025ea0aa2ae
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 56.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See abov

## RNN Data Preparation.

Converts the canonical area2_bump_canonical.h5 from area2_explore
into RNN-ready tensors for training and comparison:
  1. Load canonical H5 (spikes, rates, hand_vel, psth, condition_table)
  2. Build per-trial condition labels and integer codes
  3. Split into train / val / test using the NLB 'split' column
  4. Z-score normalize rates per neuron (fit on train set only)
  5. Create input / target pairs for multiple training objectives:
       - Decoder:   rates → hand_vel
       - Generator:  condition one-hot → rates (teacher forcing)
  6. Compute condition-averaged PSTHs per split (for RSA / DSA downstream)
  7. Save everything as a single RNN-ready H5

**Outputs**:
  - area2_bump_rnn_ready.h5         — all split tensors + metadata
  - rnn_data_prep_summary.json      — machine-readable summary

In [2]:
from __future__ import annotations

import os
import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Union

import numpy as np
import pandas as pd
import h5py



# CORE COMPONENTS: CONFIG, LOADER, SPLITTER, NORMALIZER, WRITER


# CONFIG
@dataclass
class RNNDataConfig:
    """All knobs for RNN data preparation."""
    canonical_h5: Union[str, Path] = "/kaggle/input/notebooks/abdelwhabmohamed05/area2-explore/area2_bump_canonical.h5"
    out_dir: Union[str, Path] = "/kaggle/working"

    # Original dataset info
    raw_dir: Union[str, Path] = "/kaggle/working/000127/sub-Han"
    train_prefix: str = "*train"
    split_heldout: bool = False
    bin_width_ms: int = 5
    smooth_sd_ms: int = 40
    smooth_name: str = "smth_40"
    align_field: str = "move_onset_time"
    align_range_ms: Tuple[int, int] = (-100, 500)
    split_valid_only: bool = True

    # Normalization
    norm_method: str = "zscore"

    # Train objective settings
    decoder_lag_ms: int = 40

    # Minimum trials per condition for PSTH
    min_trials_per_condition: int = 3

    def resolved_paths(self) -> Dict[str, Path]:
        out = Path(self.out_dir)
        out.mkdir(parents=True, exist_ok=True)
        return {
            "canonical_h5": Path(self.canonical_h5),
            "out_dir": out,
            "rnn_ready_h5": out / "area2_bump_rnn_ready.h5",
            "summary_json": out / "rnn_data_prep_summary.json",
        }


# CANONICAL H5 LOADER-------------------------------------------
@dataclass
class CanonicalData:
    """Container for data loaded from the canonical H5."""
    spikes: np.ndarray
    rates: np.ndarray
    hand_vel: Optional[np.ndarray]
    psth: np.ndarray
    time_bins_ms: np.ndarray 
    condition_table: np.ndarray
    pca_trajectories: np.ndarray 
    attrs: Dict

    @property
    def n_trials(self): return self.spikes.shape[0]
    @property
    def n_time(self): return self.spikes.shape[1]
    @property
    def n_neurons(self): return self.spikes.shape[2]
    @property
    def n_conditions(self): return self.psth.shape[0]


def load_canonical_h5(path: Path) -> CanonicalData:
    """Load all tensors from the canonical H5 produced by 01_area2_explore."""
    with h5py.File(path, "r") as f:
        attrs = {k: (v.tolist() if hasattr(v, "tolist") else v)
                 for k, v in f.attrs.items()}
        spikes = f["spikes"][:]
        rates = f["rates"][:]
        hand_vel = f["hand_vel"][:] if "hand_vel" in f else None
        psth = f["psth"][:]
        time_bins_ms = f["time_bins_ms"][:]
        cond_table = f["condition_table"][:] if "condition_table" in f else np.array([])
        pca_traj = f["pca_trajectories"][:] if "pca_trajectories" in f else np.array([])

    return CanonicalData(
        spikes=spikes, rates=rates, hand_vel=hand_vel,
        psth=psth, time_bins_ms=time_bins_ms,
        condition_table=cond_table, pca_trajectories=pca_traj,
        attrs=attrs,
    )


# TRIAL INFO REBUILDER
def make_trial_type_labels(trial_info: pd.DataFrame) -> pd.Series:
    """Per-trial string label: 'active_045' or 'passive_180'."""
    labels = pd.Series(index=trial_info.index, dtype="object")
    has_bump = trial_info["ctr_hold_bump"].fillna(False).astype(bool)
    cond_dir = trial_info["cond_dir"]
    active = ~has_bump & cond_dir.notna()
    passive = has_bump & cond_dir.notna()
    labels[active] = "active_" + cond_dir[active].astype(int).astype(str).str.zfill(3)
    labels[passive] = "passive_" + cond_dir[passive].astype(int).astype(str).str.zfill(3)
    return labels


def rebuild_trial_info(cfg: RNNDataConfig, n_trials: int) -> pd.DataFrame:
    """Reload NWBDataset to get trial_info with split labels.

    We split column which is not stored in the canonical H5.
    Falls back to a synthetic split if NWB data is unavailable.
    """
    try:
        from nlb_tools.nwb_interface import NWBDataset
        raw_path = Path(cfg.raw_dir)
        if raw_path.exists():
            dataset = NWBDataset(
                fpath=str(raw_path),
                prefix=cfg.train_prefix,
                split_heldout=cfg.split_heldout,
                skip_fields=[],
            )
            dataset.resample(cfg.bin_width_ms)
            ti = dataset.trial_info.copy()
            # Filter to only valid trials
            if cfg.split_valid_only:
                ti = ti[ti["split"].astype(str) != "none"].reset_index(drop=True)
            ti["trial_type_label"] = make_trial_type_labels(ti)
            # Truncate or pad to match canonical H5 trial count
            ti = ti.iloc[:n_trials].reset_index(drop=True)
            return ti
    except Exception:
        pass
    return _synthetic_trial_info(n_trials)


def _synthetic_trial_info(n_trials: int) -> pd.DataFrame:
    """Create minimal synthetic trial_info for when NWB is unavailable."""
    rng = np.random.RandomState(42)
    directions = [0, 45, 90, 135, 180, 225, 270, 315]
    rows = []
    for i in range(n_trials):
        is_bump = rng.random() < 0.5
        d = rng.choice(directions)
        split = "train" if rng.random() < 0.8 else "val"
        label = ("passive_" if is_bump else "active_") + f"{d:03d}"
        rows.append({
            "trial_id": i,
            "ctr_hold_bump": is_bump,
            "cond_dir": float(d),
            "split": split,
            "result": "R",
            "trial_type_label": label,
        })
    return pd.DataFrame(rows)


# CONDITION ENCODER
def encode_conditions(trial_info: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Encode trial_type_label as integer codes and one-hot vectors.

    Returns:
        cond_ids:     (n_trials,) int array of condition indices
        cond_onehot:  (n_trials, n_conditions) float32 one-hot
        label_order:  list of condition labels in index order
    """
    labels = trial_info["trial_type_label"].to_numpy()
    unique_labels = sorted([l for l in np.unique(labels) if isinstance(l, str) and l != ""])
    label_to_idx = {l: i for i, l in enumerate(unique_labels)}
    n_conditions = len(unique_labels)

    cond_ids = np.array([label_to_idx.get(l, -1) for l in labels], dtype=np.int32)
    cond_onehot = np.zeros((len(labels), n_conditions), dtype=np.float32)
    valid = cond_ids >= 0
    cond_onehot[valid, cond_ids[valid]] = 1.0

    return cond_ids, cond_onehot, unique_labels


# SPLITTER
@dataclass
class SplitData:
    """Container for a single train/val/test split."""
    rates: np.ndarray
    hand_vel: Optional[np.ndarray]
    spikes: np.ndarray
    cond_ids: np.ndarray
    cond_onehot: np.ndarray
    trial_indices: np.ndarray


def split_data(
    data: CanonicalData,
    trial_info: pd.DataFrame,
    cond_ids: np.ndarray,
    cond_onehot: np.ndarray,
) -> Dict[str, SplitData]:
    """Split data into train / val sets using the NLB split column."""
    splits = {}
    for split_name in ["train", "val"]:
        mask = trial_info["split"].astype(str) == split_name
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue
        splits[split_name] = SplitData(
            rates=data.rates[idx],
            hand_vel=data.hand_vel[idx] if data.hand_vel is not None else None,
            spikes=data.spikes[idx],
            cond_ids=cond_ids[idx],
            cond_onehot=cond_onehot[idx],
            trial_indices=idx,
        )
    return splits


# NORMALIZER
@dataclass
class NormStats:
    """Z-score normalization statistics (fit on train set)."""
    mean: np.ndarray
    std: np.ndarray


def compute_norm_stats(train_rates: np.ndarray, method: str = "zscore") -> NormStats:
    """Compute normalization stats from training data."""
    # Flatten trials and time: (n_trials * n_time, n_neurons)
    flat = train_rates.reshape(-1, train_rates.shape[-1])
    mean = flat.mean(axis=0).astype(np.float32)
    std = flat.std(axis=0).astype(np.float32)
    std[std < 1e-8] = 1.0
    return NormStats(mean=mean, std=std)


def apply_norm(rates: np.ndarray, stats: NormStats, method: str = "zscore") -> np.ndarray:
    """Apply normalization."""
    if method == "none":
        return rates.copy()
    return ((rates - stats.mean) / stats.std).astype(np.float32)


# DECODER INPUT/TARGET BUILDER
def build_decoder_pairs(
    rates: np.ndarray,
    hand_vel: Optional[np.ndarray],
    lag_bins: int,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """Build lagged input/target pairs for neural decoding.

    rates → hand_vel with a temporal lag.
    Returns (X, Y) where X is rates[:, :-lag, :] and Y is hand_vel[:, lag:, :].
    """
    if hand_vel is None or lag_bins <= 0:
        return rates, hand_vel
    X = rates[:, :-lag_bins, :]
    Y = hand_vel[:, lag_bins:, :]
    return X, Y


# CONDITION-AVERAGED PSTH PER SPLIT
def compute_split_psth(
    rates: np.ndarray,
    cond_ids: np.ndarray,
    n_conditions: int,
    min_trials: int = 3,
) -> np.ndarray:
    """Compute per-condition average firing rates for a given split.

    Returns: (n_conditions, n_time, n_neurons) with NaN for missing conditions.
    """
    n_time, n_neurons = rates.shape[1], rates.shape[2]
    psth = np.full((n_conditions, n_time, n_neurons), np.nan, dtype=np.float32)
    for c in range(n_conditions):
        mask = cond_ids == c
        if mask.sum() >= min_trials:
            psth[c] = rates[mask].mean(axis=0)
    return psth


# HDF5 WRITER
def save_rnn_ready_h5(
    path: Path,
    splits: Dict[str, SplitData],
    norm_stats: NormStats,
    label_order: List[str],
    canonical_psth: np.ndarray,
    split_psths: Dict[str, np.ndarray],
    cfg: RNNDataConfig,
) -> None:
    """Save all RNN-ready data to a single H5 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, "w") as f:
        # Metadata
        f.attrs["bin_width_ms"] = cfg.bin_width_ms
        f.attrs["align_field"] = cfg.align_field
        f.attrs["align_range_ms"] = cfg.align_range_ms
        f.attrs["smooth_sd_ms"] = cfg.smooth_sd_ms
        f.attrs["norm_method"] = cfg.norm_method
        f.attrs["decoder_lag_ms"] = cfg.decoder_lag_ms
        f.attrs["n_conditions"] = len(label_order)
        f.attrs["created_by"] = "neuroagents/04_rnn_data_prep.py"

        # Normalization stats
        norm_grp = f.create_group("norm_stats")
        norm_grp.create_dataset("mean", data=norm_stats.mean)
        norm_grp.create_dataset("std", data=norm_stats.std)
        f.create_dataset("condition_labels",
                         data=np.array(label_order, dtype="S"))
        f.create_dataset("canonical_psth", data=canonical_psth, compression="gzip")

        # Per-split data
        for split_name, sd in splits.items():
            grp = f.create_group(f"splits/{split_name}")
            grp.attrs["n_trials"] = sd.rates.shape[0]

            # Raw and normalized rates
            grp.create_dataset("rates_raw", data=sd.rates, compression="gzip")
            grp.create_dataset("rates_normed",
                               data=apply_norm(sd.rates, norm_stats, cfg.norm_method),
                               compression="gzip")
            grp.create_dataset("spikes", data=sd.spikes, compression="gzip")

            # Hand velocity
            if sd.hand_vel is not None:
                grp.create_dataset("hand_vel", data=sd.hand_vel, compression="gzip")

            # Condition codes
            grp.create_dataset("cond_ids", data=sd.cond_ids)
            grp.create_dataset("cond_onehot", data=sd.cond_onehot, compression="gzip")
            grp.create_dataset("trial_indices", data=sd.trial_indices)

            # Decoder pairs (lagged)
            lag_bins = int(round(cfg.decoder_lag_ms / cfg.bin_width_ms))
            X, Y = build_decoder_pairs(
                apply_norm(sd.rates, norm_stats, cfg.norm_method),
                sd.hand_vel, lag_bins
            )
            dec_grp = grp.create_group("decoder")
            dec_grp.create_dataset("X", data=X, compression="gzip")
            if Y is not None:
                dec_grp.create_dataset("Y", data=Y, compression="gzip")
            dec_grp.attrs["lag_bins"] = lag_bins

            # Split PSTH
            if split_name in split_psths:
                grp.create_dataset("psth", data=split_psths[split_name],
                                   compression="gzip")

# RNN DATA PREP DRIVER

if __name__ == "__main__":
    cfg = RNNDataConfig()
    paths = cfg.resolved_paths()
    summary: Dict = {"config": {k: str(v) if isinstance(v, Path) else v
                                for k, v in cfg.__dict__.items()}}

    # Load canonical H5
    if not paths["canonical_h5"].exists():
        raise FileNotFoundError(
            f"Canonical H5 not found at {paths['canonical_h5']}. "
            f"Run 01_area2_explore.py first."
        )
    data = load_canonical_h5(paths["canonical_h5"])
    summary["canonical"] = {
        "n_trials": data.n_trials,
        "n_time": data.n_time,
        "n_neurons": data.n_neurons,
        "n_conditions": data.n_conditions,
        "spikes_shape": list(data.spikes.shape),
        "rates_shape": list(data.rates.shape),
        "hand_vel_shape": list(data.hand_vel.shape) if data.hand_vel is not None else None,
    }

    # Rebuild trial_info with split labels
    trial_info = rebuild_trial_info(cfg, data.n_trials)
    summary["trial_info"] = {
        "n_rows": len(trial_info),
        "columns": list(trial_info.columns),
        "split_counts": trial_info["split"].value_counts().to_dict()
              if "split" in trial_info.columns else {},
    }

    # Encode conditions
    cond_ids, cond_onehot, label_order = encode_conditions(trial_info)
    summary["conditions"] = {
        "n_conditions": len(label_order),
        "labels": label_order,
        "cond_id_range": [int(cond_ids.min()), int(cond_ids.max())],
    }

    # Split data
    splits = split_data(data, trial_info, cond_ids, cond_onehot)
    summary["splits"] = {}
    for name, sd in splits.items():
        summary["splits"][name] = {
            "n_trials": sd.rates.shape[0],
            "rates_shape": list(sd.rates.shape),
            "hand_vel_shape": list(sd.hand_vel.shape) if sd.hand_vel is not None else None,
        }

    # Compute normalization stats from train set
    if "train" in splits:
        norm_stats = compute_norm_stats(splits["train"].rates, cfg.norm_method)
    else:
        norm_stats = compute_norm_stats(data.rates, cfg.norm_method)
    summary["norm_stats"] = {
        "method": cfg.norm_method,
        "mean_range": [float(norm_stats.mean.min()), float(norm_stats.mean.max())],
        "std_range": [float(norm_stats.std.min()), float(norm_stats.std.max())],
    }

    # Compute per-split PSTHs
    n_conds = len(label_order)
    split_psths = {}
    for name, sd in splits.items():
        split_psths[name] = compute_split_psth(
            apply_norm(sd.rates, norm_stats, cfg.norm_method),
            sd.cond_ids, n_conds, cfg.min_trials_per_condition,
        )
    summary["split_psths"] = {
        name: {"shape": list(p.shape), "n_valid_conditions": int(np.isfinite(p[:, 0, 0]).sum())}
        for name, p in split_psths.items()
    }

    # Save RNN-ready H5
    save_rnn_ready_h5(
        paths["rnn_ready_h5"], splits, norm_stats, label_order,
        data.psth, split_psths, cfg,
    )
    summary["rnn_ready_h5"] = str(paths["rnn_ready_h5"])
    summary["rnn_ready_h5_size_mb"] = round(
        os.path.getsize(paths["rnn_ready_h5"]) / 1e6, 2
    )

    # Write JSON summary
    with open(paths["summary_json"], "w") as f:
        json.dump(summary, f, indent=2, default=str)

    n_train = splits.get("train", SplitData(
        rates=np.array([]), hand_vel=None, spikes=np.array([]),
        cond_ids=np.array([]), cond_onehot=np.array([]),
        trial_indices=np.array([]),
    )).rates.shape[0] if "train" in splits else 0
    n_val = splits.get("val", SplitData(
        rates=np.array([]), hand_vel=None, spikes=np.array([]),
        cond_ids=np.array([]), cond_onehot=np.array([]),
        trial_indices=np.array([]),
    )).rates.shape[0] if "val" in splits else 0

    print(f"[rnn_data_prep] OK  train={n_train} val={n_val} "
          f"conditions={len(label_order)} neurons={data.n_neurons} "
          f"h5={summary['rnn_ready_h5_size_mb']}MB -> {paths['rnn_ready_h5']}")


[rnn_data_prep] OK  train=272 val=92 conditions=16 neurons=65 h5=19.64MB -> /kaggle/working/area2_bump_rnn_ready.h5
